# M2M-100 418M + LoRA **v8** — 3 ngôn ngữ, 4 chiều: ja↔vi, en↔vi

**Nền tảng (từ v7, đã chứng minh):** train từ M2M-100 gốc cả 4 chiều trong một lần; `noise_en` nhẹ cho
Parakeet-EN; bỏ 24 câu `general` lệch nghĩa vi↔ja khỏi các chiều có tiếng Nhật (`skip_ja`).
v7 đạt: ja→vi 51.6 / vi→ja 38.2 (nhiễu, giữ được v6), en→vi 56.7 / vi→en 53.6 (gốc chỉ ~37), contrast 26/28.

**Khác v7 — sửa đúng các lỗi còn thấy ở output v7:**
- `data_v8.jsonl` = data v7 (1.072 bộ ba) + **111 bộ ba mới**:
  - **Đảo nghĩa** (40): bật/tắt, mở/đóng, enable/disable, chặn/bỏ chặn — v7 dịch マイクを**オン** → "em **tắt** mic".
  - **Thuật ngữ đa nghĩa** (41): biến (≠ "thay đổi"), mount, "push" KHÔNG phải git (dời việc), ôm việc (≠ 抱擁),
    cổng bị chiếm (≠ 占領), branch ≠ chi nhánh công ty.
  - **Nhân quả "nên" vs mục đích "để"**, và "we" → mình/chúng ta (30).
- Eval: thêm cột **v7** làm mốc; kiểm **đảo nghĩa bật/tắt**; contrast set thêm 9 case v8 (tổng 37).
- `EPOCHS = 7` (v7 tốt nhất ở epoch 4, overfit từ epoch 5 — early stopping vẫn giữ).
- 103 câu eval giữ nguyên → so thẳng được với v6 và v7.

**Cách chạy:** Runtime → Change runtime type → **T4 GPU**. Upload `data_v8.jsonl` vào `/content` (hoặc để trên Drive, xem CELL 2).
**Run all** — chạy thẳng tới cuối (merge + convert CT2 + zip), ~15 phút.

**Kết quả:** `m2m100_418M_int8_v8.zip` → giải nén vào `backend/models/m2m100_418M_int8_v8/` rồi A/B ở máy local (hướng dẫn ở cell cuối).

In [ ]:
# CELL 1 — cài đặt
!pip install -q "transformers==5.14.1" "peft==0.20.0" datasets accelerate sentencepiece sacrebleu
!pip uninstall -y torchao -q

import torch, transformers, peft
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Bật GPU: Runtime -> Change runtime type -> T4 GPU"

# ⚠️ Backend (venv local) đang chạy transformers 5.14.1. Train và serve cùng một
# bản để tokenizer M2M-100 hành xử giống hệt nhau.

In [ ]:
# CELL 2 — nạp dữ liệu
import json, random, re, unicodedata, collections
from pathlib import Path

DATA_FILE = "data_v8.jsonl"
# Để file trên Drive thì bỏ comment 2 dòng dưới:
# from google.colab import drive; drive.mount("/content/drive")
# DATA_FILE = "/content/drive/MyDrive/<thư_mục>/data_v8.jsonl"

LANGS = ("vi", "ja", "en")
DIRECTIONS = [("ja", "vi"), ("vi", "ja"), ("en", "vi"), ("vi", "en")]

ALL = []
for i, line in enumerate(open(DATA_FILE, encoding="utf-8"), 1):
    line = line.strip()
    if not line:
        continue
    d = json.loads(line)
    missing = [k for k in (*LANGS, "id", "origin", "split") if not d.get(k)]
    if missing:
        raise ValueError(f"Dòng {i} thiếu {missing}: {line[:80]}")
    ALL.append({k: (v.strip() if isinstance(v, str) else v) for k, v in d.items()})

eval_rows  = [r for r in ALL if r["split"] == "eval"]
train_rows = [r for r in ALL if r["split"] == "train"]

# Không câu nguồn nào của eval được lọt vào train (ở bất kỳ ngôn ngữ nào)
for lang in LANGS:
    leak = {r[lang] for r in eval_rows} & {r[lang] for r in train_rows}
    assert not leak, f"Rò rỉ eval->train ({lang}): {list(leak)[:3]}"

print(f"Tổng {len(ALL)} bộ ba | train {len(train_rows)} | eval {len(eval_rows)}")
for o, n in sorted(collections.Counter(r["origin"] for r in train_rows).items()):
    print(f"  train/{o:16s} {n:4d}")
n_skip = sum(1 for r in train_rows if r.get("skip_ja"))
print(f"  skip_ja (vi↔ja lệch nghĩa, chỉ dùng cho en↔vi): {n_skip}")
assert not any(r.get("skip_ja") for r in eval_rows), "Eval không được có câu lệch nghĩa"
assert len(eval_rows) == 103, "Eval phải đúng 103 câu của v6 để so sánh được"

In [ ]:
# CELL 3 — mô phỏng lỗi ASR cho câu NGUỒN (câu đích luôn sạch)

# ---- JA: giữ nguyên v6. Parakeet-JA (NeMo CTC) không sinh dấu câu, không khoảng trắng.
# ー và ・ là một phần của từ -> KHÔNG xoá (bug của v4).
JA_PUNCT = re.compile(r"[。、！？!?,.「」『』（）()…―〜]")

def noise_ja(s, rng=None, level=1.0):
    s = JA_PUNCT.sub("", s)
    return re.sub(r"[\s　]+", "", s)

# ---- VI: giữ nguyên v6. Zipformer-VI: không dấu câu, chữ thường, sai dấu thanh, nhầm phụ âm, nuốt từ.
TONES   = "̣̀́̃̉"
ASCII_W = re.compile(r"^[a-z0-9._/#-]+$")
VN_CONFUS = [("d","gi"),("gi","d"),("ch","tr"),("tr","ch"),("s","x"),("x","s"),("n","l"),("l","n")]

def _corrupt_tone(w, rng):
    d = list(unicodedata.normalize("NFD", w))
    idx = [i for i, c in enumerate(d) if c in TONES]
    if not idx:
        return w
    i = rng.choice(idx)
    d[i] = "" if rng.random() < 0.45 else rng.choice([t for t in TONES if t != d[i]])
    return unicodedata.normalize("NFC", "".join(d))

def noise_vi(s, rng=None, level=1.0):
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"[.,!?;:\"'’“”()\[\]\-–—…]", " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    if not level or rng is None:
        return s
    words, out = s.split(), []
    for w in words:
        if ASCII_W.match(w):
            out.append(w); continue
        r = rng.random()
        if r < 0.04 * level and len(words) > 6:
            continue
        elif r < 0.24 * level:
            w = _corrupt_tone(w, rng)
        elif r < 0.30 * level:
            for a, b in VN_CONFUS:
                if w.startswith(a):
                    w = b + w[len(a):]; break
        out.append(w)
    return " ".join(out)

# ---- EN: MỚI. Parakeet-EN (TDT 0.6B v2) xuất câu gần như sạch: có dấu câu, viết hoa,
# chữ số. Đo trên backend: CER 0.005 với audio sạch; lỗi định dạng hay gặp nhất là
# "4:30" -> "4.30". Nên nhiễu NHẸ — làm nhiễu kiểu noise_vi sẽ dạy model sai phân bố.
def noise_en(s, rng=None, level=1.0):
    s = unicodedata.normalize("NFC", s).strip()
    if not level or rng is None:
        return s
    if rng.random() < 0.5 * level:
        s = re.sub(r"\b(\d{1,2}):(\d{2})\b", r"\1.\2", s)     # 4:30 -> 4.30
    if rng.random() < 0.25 * level:
        s = s.replace(",", "")                               # mất dấu phẩy
    if rng.random() < 0.25 * level:
        s = s.rstrip(".!?")                                  # mất dấu chấm cuối
    if rng.random() < 0.10 * level:
        s = s.lower()
    words = s.split()
    if len(words) > 8 and rng.random() < 0.08 * level:       # nuốt một từ ngắn
        cand = [i for i, w in enumerate(words) if len(w) <= 3]
        if cand:
            del words[rng.choice(cand)]
            s = " ".join(words)
    return s

NOISE = {"ja": noise_ja, "vi": noise_vi, "en": noise_en}

_r = random.Random(3)
for r in train_rows[:2] + [x for x in train_rows if x["origin"] == "target_time"][:2]:
    print(f"en gốc : {r['en']}\n   nhiễu: {noise_en(r['en'], _r, 1.0)}")
    print(f"vi gốc : {r['vi']}\n   nhiễu: {noise_vi(r['vi'], _r, 1.0)}")
    print(f"ja nhiễu: {noise_ja(r['ja'])}\n")

In [ ]:
# CELL 4 — bộ ba -> dòng train/eval theo 4 chiều
from datasets import Dataset

# Lặp lại dữ liệu targeted bao nhiêu lần. 1 = không lặp. Tăng lên 2 nếu contrast
# set tiếng Anh (CELL 11) vẫn trượt case sáng/chiều sau khi train.
TARGET_REPEAT = 1

def to_rows(rows, rng=None, levels=(0.0,)):
    out = []
    for r in rows:
        for s, t in DIRECTIONS:
            # skip_ja: cặp vi↔ja gốc lệch nghĩa (phụ đề ghép tự động). Bản en dịch
            # theo vế vi nên vẫn dùng cho en↔vi, chỉ bỏ các chiều có tiếng Nhật.
            if r.get("skip_ja") and "ja" in (s, t):
                continue
            lv = rng.choice(levels) if rng else levels[0]
            out.append({"src_text": NOISE[s](r[s], rng, lv), "tgt_text": r[t],
                        "src_lang": s, "tgt_lang": t, "level": lv, "origin": r["origin"]})
    return out

base = train_rows + [r for r in train_rows if r["origin"].startswith("target_")] * (TARGET_REPEAT - 1)
rng = random.Random(0)
train_list = to_rows(base, rng, levels=(0.0, 0.5, 1.0))     # sạch / phòng yên / phòng ồn
random.Random(1).shuffle(train_list)

train_dataset    = Dataset.from_list(train_list)
eval_dataset_raw = Dataset.from_list(to_rows(eval_rows, None, levels=(0.0,)))
eval_noisy_raw   = Dataset.from_list(to_rows(eval_rows, random.Random(99), levels=(1.0,)))

print(f"📦 Train {len(train_dataset)} dòng | Eval sạch {len(eval_dataset_raw)} | Eval nhiễu {len(eval_noisy_raw)}")
print("Theo chiều:", dict(collections.Counter(f"{r['src_lang']}->{r['tgt_lang']}" for r in train_list)))
print("Theo mức nhiễu:", dict(collections.Counter(r["level"] for r in train_list)))

In [ ]:
# CELL 5 — model + LoRA
import torch
from transformers import (M2M100ForConditionalGeneration, M2M100Tokenizer,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, TaskType

model_id = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_id)
model = M2M100ForConditionalGeneration.from_pretrained(model_id)

# v6: r=24. v7 học gấp đôi số chiều (thêm en↔vi) -> tăng dung lượng adapter lên r=32.
# Adapter vẫn chỉ ~3% tham số; model sau merge có kích thước y như cũ.
lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

MAX_LEN = 128

def preprocess(ex):
    # M2M-100: set src_lang/tgt_lang TRƯỚC khi encode (chèn token ngôn ngữ)
    tokenizer.src_lang = ex["src_lang"]
    mi = tokenizer(ex["src_text"], max_length=MAX_LEN, truncation=True)
    tokenizer.tgt_lang = ex["tgt_lang"]
    mi["labels"] = tokenizer(text_target=ex["tgt_text"], max_length=MAX_LEN,
                             truncation=True)["input_ids"]
    return mi

tokenized_train      = train_dataset.map(preprocess,    remove_columns=train_dataset.column_names)
tokenized_eval       = eval_dataset_raw.map(preprocess, remove_columns=eval_dataset_raw.column_names)
tokenized_eval_noisy = eval_noisy_raw.map(preprocess,   remove_columns=eval_noisy_raw.column_names)

# PeftModel bọc ngoài làm DataCollatorForSeq2Seq không thấy
# prepare_decoder_input_ids_from_labels -> đưa base model cho collator.
collator_model = model.get_base_model() if hasattr(model, "get_base_model") else model
data_collator = DataCollatorForSeq2Seq(tokenizer, model=collator_model)
print(f"Train {len(tokenized_train)} | Eval sạch {len(tokenized_eval)} | Eval nhiễu {len(tokenized_eval_noisy)}")

In [ ]:
# CELL 6 — kiểm tra forward/backward trước khi train
from torch.utils.data import DataLoader

dev = "cuda"
model.to(dev)
batch = next(iter(DataLoader(tokenized_train.select(range(4)), batch_size=2, collate_fn=data_collator)))
print("Khoá trong batch:", {k: tuple(v.shape) for k, v in batch.items()})
model.train()
out = model(**{k: v.to(dev) for k, v in batch.items()})
out.loss.backward(); model.zero_grad(set_to_none=True)
print(f"✅ forward + backward OK — loss = {out.loss.item():.4f}")

In [ ]:
# CELL 7 — train
EPOCHS = 7          # v7 tốt nhất ở epoch 4; early stopping dừng sớm khi eval_loss hết giảm
BS = 16             # T4 dư sức với 418M + LoRA ở MAX_LEN 128
steps_per_epoch = -(-len(tokenized_train) // BS)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(10, int(total_steps * 0.06))
print(f"steps/epoch {steps_per_epoch} | tối đa {total_steps} | warmup {warmup_steps}")

args = Seq2SeqTrainingArguments(
    output_dir="./m2m418-lora-v8",
    learning_rate=3e-4,              # v6: 2.5e-4 @ BS 8. Batch gấp đôi -> nâng nhẹ
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,
    # KHÔNG bật label_smoothing_factor: Trainer pop 'labels' -> M2M100 không dựng
    # được decoder_input_ids -> ValueError (đã gặp ở v5).
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval_noisy,      # chọn checkpoint theo eval NHIỄU, cả 4 chiều
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

In [ ]:
# CELL 8 — lưu adapter + đường loss
import os, matplotlib.pyplot as plt, numpy as np

os.makedirs("./adapter_best_v8", exist_ok=True)
model.save_pretrained("./adapter_best_v8")
tokenizer.save_pretrained("./adapter_best_v8")
json.dump(trainer.state.log_history, open("./adapter_best_v8/log_history.json", "w"),
          ensure_ascii=False, indent=1)
print("✅ adapter_best_v8 đã lưu")
# Muốn giữ khi Colab reset:
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.make_archive("/content/drive/MyDrive/m2m418_adapter_v8", "zip", "./adapter_best_v8")

h = trainer.state.log_history
tr = [(x["epoch"], x["loss"]) for x in h if "loss" in x and "eval_loss" not in x]
ev = [(x["epoch"], x["eval_loss"]) for x in h if "eval_loss" in x]
plt.figure(figsize=(7, 3.5))
if tr: plt.plot(*zip(*tr), label="train", alpha=.7)
if ev: plt.plot(*zip(*ev), label="eval (nhiễu)", marker="s", lw=2)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3); plt.show()
for e, l in ev:
    print(f"  epoch {e:.0f}: {l:.4f}{'  ← best' if l == min(y for _, y in ev) else ''}")

In [ ]:
# CELL 9 — chrF cho 4 chiều: v8 vs M2M-100 GỐC (tắt adapter) vs mốc v6, v7
import sacrebleu, numpy as np, torch

RESULTS = {}

@torch.no_grad()
def gen_batch(texts, src, tgt, bs=32):
    outs = []; model.eval()
    for i in range(0, len(texts), bs):
        tokenizer.src_lang = src
        enc = tokenizer(texts[i:i+bs], return_tensors="pt", padding=True,
                        truncation=True, max_length=128).to(model.device)
        o = model.generate(**enc, forced_bos_token_id=tokenizer.get_lang_id(tgt),
                           max_length=128, num_beams=4, early_stopping=True,
                           no_repeat_ngram_size=3, repetition_penalty=1.05)
        outs += tokenizer.batch_decode(o, skip_special_tokens=True)
    return outs

def boot_ci(sent, n=1000, seed=0):
    r = np.random.default_rng(seed); a = np.asarray(sent)
    m = [a[r.integers(0, len(a), len(a))].mean() for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

def score(ds, label, tag="v8"):
    print(f"\n===== {tag} / {label} =====")
    for src, tgt in DIRECTIONS:
        sub = [r for r in ds if r["src_lang"] == src and r["tgt_lang"] == tgt]
        srcs = [r["src_text"] for r in sub]; refs = [r["tgt_text"] for r in sub]
        preds = gen_batch(srcs, src, tgt)
        chrf = sacrebleu.corpus_chrf(preds, [refs]).score
        sent = [sacrebleu.sentence_chrf(p, [r]).score for p, r in zip(preds, refs)]
        lo, hi = boot_ci(sent)
        RESULTS[(tag, label, src, tgt)] = dict(chrf=chrf, sent=sent, srcs=srcs, refs=refs,
                                               preds=preds, ci=(lo, hi))
        print(f"  {src}→{tgt}: chrF {chrf:5.2f} | CI95 câu [{lo:5.1f}, {hi:5.1f}] (n={len(sub)})")

score(eval_dataset_raw, "SẠCH")
score(eval_noisy_raw,   "NHIỄU")
with model.disable_adapter():                 # cùng model, tắt LoRA = M2M-100 gốc
    score(eval_noisy_raw, "NHIỄU", tag="gốc")

# Mốc (cùng 103 câu eval). v6 chép từ notebook v6 (không có en); v7 chép từ lần chạy v7.
V6 = {("NHIỄU", "ja", "vi"): 49.53, ("NHIỄU", "vi", "ja"): 38.92}
V7 = {("NHIỄU", "ja", "vi"): 51.63, ("NHIỄU", "vi", "ja"): 38.21,
      ("NHIỄU", "en", "vi"): 56.70, ("NHIỄU", "vi", "en"): 53.57,
      ("SẠCH", "ja", "vi"): 51.63, ("SẠCH", "vi", "ja"): 49.82,
      ("SẠCH", "en", "vi"): 56.28, ("SẠCH", "vi", "en"): 63.89}
nan = float("nan")

print("\n" + "=" * 84)
print(f"{'chiều':8s}{'gốc':>8s}{'v6':>8s}{'v7':>8s}{'v8':>8s}{'v8 sạch':>9s}{'(v7 sạch)':>11s}   đánh giá (cột nhiễu)")
print("=" * 84)
for src, tgt in DIRECTIONS:
    g  = RESULTS[("gốc", "NHIỄU", src, tgt)]["chrf"]
    n8 = RESULTS[("v8", "NHIỄU", src, tgt)]
    c8 = RESULTS[("v8", "SẠCH", src, tgt)]["chrf"]
    v6, v7 = V6.get(("NHIỄU", src, tgt), nan), V7[("NHIỄU", src, tgt)]
    half = (n8["ci"][1] - n8["ci"][0]) / 2
    # So với bản TỐT NHẤT trước đó; chênh trong nửa CI = ngang (103 câu không đọc được chênh nhỏ hơn)
    best = max(v for v in (v6, v7) if v == v)
    d = n8["chrf"] - best
    verdict = ("✅ tốt hơn" if d > half else "✅ ngang" if d >= -half else f"🔴 tụt quá CI ({d:+.1f})")
    print(f"{src}→{tgt:5s}{g:8.2f}{v6:8.2f}{v7:8.2f}{n8['chrf']:8.2f}{c8:9.2f}{V7[('SẠCH', src, tgt)]:11.2f}   {verdict}")
print("\nv8 không được tụt quá CI so với bản tốt nhất trước đó. Cột NHIỄU mới phản ánh app thật.")

In [ ]:
# CELL 10 — kiểm tra lỗi cụ thể: register vi→ja, sáng/chiều, con số
POLITE = re.compile(r"(です|ます|ません|ました|でした|ください|ましょう|ございます"
                    r"|ますか|ですか|ましたか|ませんか|ましょうか|でしょうか)[。！？]*$")

print("(1) REGISTER vi→ja — % output ở thể lịch sự (v6: 98.1% trên eval nhiễu)")
for label in ("SẠCH", "NHIỄU"):
    p = RESULTS[("v8", label, "vi", "ja")]["preds"]
    pol = 100 * sum(1 for x in p if POLITE.search(x.strip())) / len(p)
    print(f"  {label:6s} {pol:5.1f}%  {'✅' if pol >= 90 else '🔴'}")

_AM = {"en": r"\ba\.?m\.?\b|\bmorning\b", "ja": r"午前|朝", "vi": r"\bsáng\b"}
_PM = {"en": r"\bp\.?m\.?\b|\bafternoon\b|\bevening\b|\btonight\b",
       "ja": r"午後|夕方|夜", "vi": r"\bchiều\b|\btối\b"}
def ampm(t, lang):
    a = bool(re.search(_AM[lang], t, re.I)); p = bool(re.search(_PM[lang], t, re.I))
    return None if a == p else ("am" if a else "pm")

def numbers(t):
    t = unicodedata.normalize("NFKC", t)
    return {re.sub(r"[.,](?=\d{3}\b)", "", m) for m in re.findall(r"\d+(?:[.,]\d{3})*", t)}

print("\n(2) SÁNG/CHIỀU bị đảo và SỐ bị mất — trên eval sạch")
for src, tgt in DIRECTIONS:
    r = RESULTS[("v8", "SẠCH", src, tgt)]
    flip = sum(1 for p, ref in zip(r["preds"], r["refs"])
               if ampm(ref, tgt) and ampm(p, tgt) and ampm(ref, tgt) != ampm(p, tgt))
    lost = sum(1 for p, ref in zip(r["preds"], r["refs"])
               if numbers(ref) and not numbers(ref) <= numbers(p))
    print(f"  {src}→{tgt}: sáng/chiều đảo {flip} | mất/sai số {lost}")
print("  (eval 103 câu ít câu giờ giấc — contrast set ở CELL 11 mới đo trúng lỗi này)")

# (3) ĐẢO NGHĨA bật/tắt — lỗi nguy hiểm nhất v7 để lộ: マイクをオン -> "em tắt mic"
_ON  = {"vi": r"\b(bật|mở|kích hoạt|enable|cho phép)\b", "ja": r"(オン|有効|開け|許可)",
        "en": r"\b(turn(ed|s)? on|enable|open|allow|unmute)"}
_OFF = {"vi": r"\b(tắt|đóng|vô hiệu|disable|chặn)\b", "ja": r"(オフ|無効|閉じ|切っ|切ら|止め|停止)",
        "en": r"\b(turn(ed|s)? off|disable|close|block|mute)"}
def polarity(t, lang):
    on = bool(re.search(_ON[lang], t, re.I)); off = bool(re.search(_OFF[lang], t, re.I))
    return None if on == off else ("on" if on else "off")

print("\n(3) ĐẢO NGHĨA bật/tắt — trên eval sạch + nhiễu")
for src, tgt in DIRECTIONS:
    n = 0
    for label in ("SẠCH", "NHIỄU"):
        r = RESULTS[("v8", label, src, tgt)]
        n += sum(1 for p, ref in zip(r["preds"], r["refs"])
                 if polarity(ref, tgt) and polarity(p, tgt) and polarity(ref, tgt) != polarity(p, tgt))
    print(f"  {src}→{tgt}: {n} câu đảo nghĩa  {'✅' if n == 0 else '🔴'}")

In [ ]:
# CELL 11 — CONTRAST SET: 16 case v6 + 12 case tiếng Anh (v7) + 9 case v8
# (nguồn, src, tgt, PHẢI có, KHÔNG được có, mô tả)
# ⚠️ Câu nguồn KHÔNG được trùng câu nào trong data train — ô dưới tự kiểm tra.
CONTRAST = [
 # ---- v6 (giữ nguyên để so) ----
 ("Phần này ai rảnh nhận giúp không, em đang ôm hơi nhiều.", "vi", "ja", r"(手伝|助け)", r"抱擁", "ôm việc ≠ 抱擁"),
 ("Mọi người đồng ý thì mình tiến hành luôn nhé.", "vi", "ja", r"(ましょう)", r"てください", "rủ rê ≠ ra lệnh"),
 ("Cổng 8000 đang bị chiếm.", "vi", "ja", r"(使用中|使われて|使用され|埋まって)", r"占領", "port bận ≠ chiếm đóng"),
 ("Phân quyền chưa chặt, user thường vào được trang admin.", "vi", "ja", r"権限", r"特権", "権限 ≠ 特権"),
 ("Do đổi yêu cầu giữa chừng nên tiến độ chung bị ảnh hưởng.", "vi", "ja", r"(途中|変更)", r"中間要件", "đổi giữa chừng"),
 ("9 giờ sáng mai họp standup nhé mọi người.", "vi", "ja", r"9時", r"ごろ", "không thêm 'khoảng' vào giờ họp"),
 ("Mình bàn lại deadline vào chiều mai nhé.", "vi", "ja", r"(ましょう|ませんか)", r"しろ", "giữ thể lịch sự"),
 ("少し抱えすぎているので、余裕がある人に手伝ってほしいです。", "ja", "vi", r"(giúp|hỗ trợ)", r"muốn giúp đỡ", "~てほしい"),
 ("権限制御が甘く、一般ユーザーが管理画面に入れてしまいます。", "ja", "vi", r"(phân quyền|quyền)", r"mềm", "甘い = lỏng lẻo"),
 ("この件は明日までに対応していただけますか。", "ja", "vi", r"(giúp|nhờ|được không|nhé)", r"^Tôi sẽ", "nhờ vả ≠ tự nhận việc"),
 ("サーバーのメモリーが足りないので、スケールアップしてください。", "ja", "vi", r"(memory|bộ nhớ|RAM)", r"kỷ niệm", "メモリー = bộ nhớ"),
 ("レビューのコメントを反映しました。", "ja", "vi", r"(review|comment|nhận xét|góp ý)", r"bình luận phim", "review code"),
 ("恐れ入りますが、こちらの内容を来週までにご確認いただくことは可能でしょうか。", "ja", "vi",
  r"(giúp|anh|chị|được không|nhé|ạ)", r"^(Tôi|Em) sẽ", "kính ngữ lồng でしょうか"),
 ("Chị copy giúp em file config đó sang server mới với.", "vi", "ja", r"(コピー|していただけ)", r"手伝って", "giúp + động từ cụ thể"),
 ("Nếu cả nhóm không ai có ý kiến khác thì mình chốt kế hoạch này nhé.", "vi", "ja", r"(ましょう)", r"てください", "rủ rê tập thể"),
 ("Tuyệt đối không được tắt server trong giờ làm việc, nhớ kỹ điều này.", "vi", "ja",
  r"(ないでください|てください)", r"ましょう", "lệnh cấm ≠ rủ rê"),
 # ---- v7: tiếng Anh — nhắm đúng lỗi đo được trên model v6 ----
 ("Our call with the client is at 2 p.m. on Thursday.", "en", "vi", r"chiều", r"sáng", "p.m. = chiều"),
 ("Chiều nay 4 giờ mình họp lại nhé.", "vi", "en", r"(4 ?p\.?m|afternoon)", r"(\ba\.?m\b|morning)", "chiều = p.m."),
 ("9 giờ tối thứ Bảy mình bảo trì database.", "vi", "en", r"(9 ?p\.?m|night|evening)", r"(\b9 ?a\.?m|morning)", "tối = p.m."),
 ("Thanks for all your hard work this week, everyone.", "en", "vi", r"(cảm ơn|vất vả)", r"hạnh phúc", "cảm ơn cuối tuần"),
 ("Sorry for being late, my previous meeting ran over.", "en", "vi", r"xin lỗi", r"chạy", "ran over ≠ chạy"),
 ("I pushed the hotfix to the dev branch.", "en", "vi", r"push", r"(đẩy|ấn|nhấn)", "push giữ thuật ngữ"),
 ("The payment server has been down since this morning.", "en", "vi",
  r"(sập|down|ngừng|không hoạt động|lỗi)", r"giảm", "server down ≠ giảm"),
 ("The migration will cost around 1.5 million yen.", "en", "vi", r"1[.,]5 triệu", r"\b15 triệu", "1.5 million"),
 ("Chị xem giúp em cái log này với.", "vi", "en", r"(could you|can you|please)", r"see how", "nhờ xem giúp"),
 ("Chiều nay em hỏi lại team rồi phản hồi cho chị nhé.", "vi", "en",
  r"(get back|let you know|update you|reply|respond|tell you|answer)", r"\breturn", "phản hồi ≠ return"),
 ("Cổng 3000 đang bị process khác chiếm.", "vi", "en", r"(in use|used|taken|occupied|busy)", r"(seiz|captur|invad)", "port bận"),
 ("Nếu không ai phản đối thì mình merge luôn nhé.", "vi", "en", r"(let's|let us|we('ll| will| can)|go ahead)", r"^Please", "rủ rê ≠ ra lệnh"),
 # ---- v8: nhắm lỗi thấy ở output v7 (câu KHÁC data train v8) ----
 ("発言するときはマイクをオンにしてください。", "ja", "vi", r"bật", r"tắt", "オン = bật (v7 dịch thành tắt)"),
 ("Tắt thông báo email cho tài khoản test giúp em.", "vi", "ja", r"(オフ|無効|停止|止め)", r"(オン|有効に)", "tắt = オフ"),
 ("Nhớ bật lại firewall sau khi test xong nhé.", "vi", "en", r"(turn .*on|enable|re-?enable|switch .*on)", r"(turn .*off|disable)", "bật lại = turn back on"),
 ("Please disable the old API key after the migration.", "en", "vi", r"(tắt|vô hiệu|disable|hủy|huỷ|xoá|xóa|thu hồi)", r"(bật|kích hoạt|enable)", "disable ≠ bật"),
 ("Could you rename this variable?", "en", "vi", r"biến", r"tên thay đổi", "variable = biến"),
 ("Let's push the release to next Friday.", "en", "vi", r"(dời|lùi|hoãn|chuyển)", r"push", "push lịch = dời"),
 ("The server crashed, so the job didn't finish.", "en", "vi", r"(nên|vì|do)", r"để (khỏi|không)", "so = nên (nhân quả)"),
 ("Anh ấy đang ôm nhiều dự án quá.", "vi", "ja", r"抱え", r"(抱擁|抱きし)", "ôm dự án = 抱える"),
 ("Cổng 9000 đang bị chiếm, em đổi cổng khác nhé.", "vi", "ja", r"(使用中|使われて|使用され|埋まって|占有)", r"占領", "cổng bị chiếm ≠ 占領"),
]

train_src = {r[l] for r in train_rows for l in LANGS}
dup = [c[0] for c in CONTRAST if c[0] in train_src]
assert not dup, f"Contrast trùng data train (chỉ đo trí nhớ): {dup}"

def run_contrast(tag):
    passed, lines = 0, []
    for srct, s, t, must, mustnt, desc in CONTRAST:
        out = gen_batch([srct], s, t)[0]
        ok = bool(re.search(must, out, re.I)) and not re.search(mustnt, out, re.I)
        passed += ok
        lines.append((ok, s, t, desc, srct, out))
    return passed, lines

p7, lines7 = run_contrast("v8")
with model.disable_adapter():
    pg, _ = run_contrast("gốc")

for ok, s, t, desc, srct, out in lines7:
    print(f"  {'✅' if ok else '🔴'} [{s}→{t}] {desc}")
    if not ok:
        print(f"       nguồn: {srct}\n       model: {out}")
groups = [("v6", 0, 16, "v6: 14/16, v7: 14/16"), ("v7-en", 16, 28, "v7: 12/12"), ("v8", 28, len(CONTRAST), "mới")]
print(f"\n  v8 ĐẠT {p7}/{len(CONTRAST)}  (M2M-100 gốc: {pg}/{len(CONTRAST)})")
for name, a, b, ref in groups:
    print(f"    phần {name:6s}: {sum(ok for ok, *_ in lines7[a:b])}/{b - a}   (mốc {ref})")

In [ ]:
# CELL 12 — soi nhanh: câu tệ nhất mỗi chiều + dịch thử kiểu ASR
print("=" * 72); print("3 CÂU TỆ NHẤT MỖI CHIỀU (eval nhiễu)"); print("=" * 72)
for src, tgt in DIRECTIONS:
    r = RESULTS[("v8", "NHIỄU", src, tgt)]
    worst = sorted(zip(r["sent"], r["srcs"], r["refs"], r["preds"]))[:3]
    for sc, a, b, c in worst:
        print(f"[{src}→{tgt}] chrF={sc:.1f}\n  nguồn: {a}\n  chuẩn: {b}\n  model: {c}")

@torch.no_grad()
def dich(text, src, tgt):
    return gen_batch([text], src, tgt)[0]

print("\n" + "=" * 72); print("DỊCH THỬ"); print("=" * 72)
for t, s, g in [("The meeting starts at 3 p.m. tomorrow.", "en", "vi"),
                ("Can we move the review to 4.30 in the afternoon", "en", "vi"),
                ("I'll check it and get back to you later.", "en", "vi"),
                ("anh xem giúp em file này nhé", "vi", "en"),
                ("cảm ơn mọi người hôm nay vất vả rồi", "vi", "en"),
                ("明日の会議は午後3時からです", "ja", "vi"),
                ("em gửi tài liệu rồi anh xem giúp em nhé", "vi", "ja"),
                ("カメラをオフにして、マイクだけオンにしてください", "ja", "vi"),
                ("It's fine to move this task to the next sprint.", "en", "vi"),
                ("tắt chế độ debug rồi mới deploy nhé", "vi", "en")]:
    print(f"  [{s}→{g}] {t}\n          → {dich(t, s, g)}")

In [ ]:
# CELL 13 — merge LoRA vào model gốc
merged = model.merge_and_unload()        # sau lệnh này `model` không dùng lại được
merged.save_pretrained("m2m418-it-merged-v8")
tokenizer.save_pretrained("m2m418-it-merged-v8")
for f in sorted(os.listdir("m2m418-it-merged-v8")):
    print(f"  {f:32s} {os.path.getsize(f'm2m418-it-merged-v8/{f}')/1e6:8.2f} MB")

In [ ]:
# CELL 14 — convert sang CTranslate2 int8 (định dạng backend đang dùng)
!pip install -q ctranslate2
import subprocess

SRC, OUT = "m2m418-it-merged-v8", "m2m100_418M_int8_v8"
CANDIDATES = ["sentencepiece.bpe.model", "vocab.json", "tokenizer_config.json",
              "special_tokens_map.json", "added_tokens.json", "tokenizer.json"]
present = [f for f in CANDIDATES if os.path.exists(f"{SRC}/{f}")]
print("copy :", present)
if "tokenizer_config.json" not in present or "sentencepiece.bpe.model" not in present:
    raise RuntimeError("Thiếu file tokenizer — backend sẽ không nạp được (chạy lại CELL 13).")
subprocess.run(["ct2-transformers-converter", "--model", SRC, "--output_dir", OUT,
                "--quantization", "int8", "--force", "--copy_files", *present], check=True)
print("✅ Convert xong")

In [ ]:
# CELL 15 — test bản CT2 trên CPU (y như backend) + đóng gói
import ctranslate2
from transformers import AutoTokenizer

lack = [f for f in ["model.bin", "config.json", "shared_vocabulary.json"] if not os.path.exists(f"{OUT}/{f}")]
if lack:
    raise RuntimeError(f"Thiếu file lõi {lack} — convert hỏng.")

tok = AutoTokenizer.from_pretrained(OUT, local_files_only=True)
ct2 = ctranslate2.Translator(OUT, device="cpu", compute_type="int8")
LEN_MULT = {("ja", "vi"): 4.0, ("vi", "ja"): 2.0, ("en", "vi"): 3.0, ("vi", "en"): 2.0}   # = backend/main.py

def dich_ct2(text, src, tgt):
    tok.src_lang = src
    st = tok.convert_ids_to_tokens(tok.encode(text.lower() if src == "vi" else text))
    res = ct2.translate_batch([st], target_prefix=[[f"__{tgt}__"]], beam_size=4,
                              max_decoding_length=min(128, int(len(st) * LEN_MULT[(src, tgt)]) + 12),
                              repetition_penalty=1.1, no_repeat_ngram_size=3)
    return tok.decode(tok.convert_tokens_to_ids(res[0].hypotheses[0][1:]), skip_special_tokens=True).strip()

print("--- CT2 INT8 trên CPU ---")
for t, s, g in [("この部分は納品前に再確認が必要です", "ja", "vi"),
                ("cổng 8000 đang bị chiếm", "vi", "ja"),
                ("The staging server has been down since 9 a.m.", "en", "vi"),
                ("chiều nay 3 giờ mình review code nhé", "vi", "en")]:
    print(f"  [{s}→{g}] {t}\n            → {dich_ct2(t, s, g)}")

!zip -qr {OUT}.zip {OUT}
print(f"\n📦 {OUT}.zip  {os.path.getsize(f'{OUT}.zip')/1e6:.1f} MB — tải về từ panel Files bên trái.")
# Hoặc chép sang Drive:
# import shutil; shutil.copy(f"{OUT}.zip", "/content/drive/MyDrive/")

## Bước tiếp theo ở máy local (A/B với v6 đang chạy)

1. Giải nén zip vào `backend/models/m2m100_418M_int8_v8/` (**không** ghi đè `m2m100_418M_int8/`).
2. Đo cùng một bộ câu cho cả hai bản (PowerShell, trong `backend/`, venv đã activate):
   ```powershell
   python eval_mt.py --out eval_v6.csv
   $env:KATOBA_MT_DIR = "models\m2m100_418M_int8_v8"; python eval_mt.py --out eval_v7.csv; Remove-Item Env:KATOBA_MT_DIR
   ```
   `eval_mt.py` chạy đúng pipeline của app (hậu xử lý + glossary). Mốc v6 hiện tại: ja→vi 50.2, vi→ja 39.0, en→vi 49.5, vi→en 59.6 (chrF), sáng/chiều đảo 2 câu.
   ⚠️ `testdata/cau_test_mt.tsv` có vài câu cùng chủ đề với data targeted (giờ giấc, cảm ơn) nên số en sẽ hơi lạc quan — CELL 9 (103 câu eval) mới là mốc quyết định.
3. Chạy thử app thật với `$env:KATOBA_MT_DIR` như trên.
4. Tốt hơn thật → đổi tên thư mục thành `m2m100_418M_int8`, sửa `MODEL_VERSION = "v8"` trong `main.py`.